# Titanic Kaggle Challenge

Required few steps before running this:

```py
python -m pip install scikit-learn
python -m pip install seaborn
python -m pip isntall matplotlib
python -m pip install kaggle
```

---

Challenge: https://www.kaggle.com/competitions/titanic/overview \
Datasets: https://www.kaggle.com/competitions/titanic/data \

---

Written by Mohd Zaid (mdzaid0205@gmail.com)

In [ ]:
CONTESTS:str = "titanic"
LOCATION:str = "./tmp/data"
!kaggle competitions download -c {CONTESTS} -p {LOCATION}
!mkdir "{LOCATION}/{CONTESTS}"
!tar -xf "{LOCATION}/{CONTESTS}.zip" -C "{LOCATION}/{CONTESTS}"

## Basic Imports and Loading File

In [ ]:
import pandas as pd, numpy as np, os
from matplotlib import pyplot as plt
from pathlib import Path
from typing import Optional

In [ ]:
OVERWRITE_TESTS_PATH: Optional[Path] = None     # Path("Path")  # tests.csv path
OVERWRITE_TRAIN_PATH: Optional[Path] = None     # Path("Path")  # train.csv path
OVERWRITE_FINAL_PATH: Optional[Path] = None     # Path("Path")  # output path

base_train_directory:Path = Path(LOCATION) / Path(CONTESTS)
base_tests_directory:Path = Path(LOCATION) / Path(CONTESTS)

if OVERWRITE_TRAIN_PATH and OVERWRITE_TRAIN_PATH.is_file():
    base_train_directory = OVERWRITE_TRAIN_PATH.parent.name

if OVERWRITE_TESTS_PATH and OVERWRITE_TESTS_PATH.is_file():
    base_tests_directory = OVERWRITE_TESTS_PATH.parent.name

TRAIN_FILENAME:str = 'train.csv'
TESTS_FILENAME:str = 'test.csv'

train_filepath:Path = Path(TRAIN_FILENAME)
tests_filepath:Path = Path(TESTS_FILENAME)

def find_filepath(required_filename:str, base_directory:Path) -> Path:
    for (directory, _, filenames) in os.walk(base_directory):
        if required_filename in filenames:
            return Path(directory) / Path(required_filename)
    return base_directory / Path(required_filename)

train_filepath:Path = find_filepath(TRAIN_FILENAME, base_train_directory)
tests_filepath:Path = find_filepath(TESTS_FILENAME, base_tests_directory)
final_filepath:Path = Path(LOCATION) / Path(CONTESTS) / 'submission.csv'

assert train_filepath.is_file() and train_filepath.name == TRAIN_FILENAME, f"{train_filepath=} is not valid file for training"
assert tests_filepath.is_file() and tests_filepath.name == TESTS_FILENAME, f"{tests_filepath=} is not valid file for testing"

print(f"train_file  = {train_filepath}")
print(f"tests_file  = {tests_filepath}")
print(f"output_file = {final_filepath}")

In [ ]:
TRAINING_DATA:Optional[pd.DataFrame] = None
TESTINGS_DATA:Optional[pd.DataFrame] = None

if os.path.exists(train_filepath) and train_filepath.is_file():
    TRAINING_DATA = pd.read_csv(train_filepath)
if os.path.exists(tests_filepath) and tests_filepath.is_file():
    TESTINGS_DATA = pd.read_csv(tests_filepath)

assert TRAINING_DATA is not None, "Please re-check training file path"
assert TESTINGS_DATA is not None, "Please re-check testings file path"

display(TRAINING_DATA.head(5))
display(TESTINGS_DATA.head(5))


In [ ]:
display(TRAINING_DATA.info() if TRAINING_DATA is not None else "N/A")
display(TESTINGS_DATA.info() if TESTINGS_DATA is not None else "N/A")

## Data Analysis

In [ ]:
if TRAINING_DATA is not None:
    display(TRAINING_DATA.Survived.describe())

In [ ]:
target_col:str = "Survived"
noneed_col:list[str] = ["PassengerId", "Name", "Ticket"]    # "Cabin" too many NaN in cabin
params_col:list[str] = []

if TESTINGS_DATA is not None:
    for columns in TESTINGS_DATA.columns.to_list():
        if columns not in [target_col] + noneed_col:
            params_col.append(columns)

print("Target Columns:", target_col)
print("NoNeed Columns:", noneed_col)
print("Params Columns:", params_col)

In [ ]:
if TRAINING_DATA is not None:
    for columns in params_col:
        display(TRAINING_DATA[columns].describe())
        display("---"*10)

In [ ]:
def preprocess(df:Optional[pd.DataFrame], target_col:str, params_col:list[str], noneed_col:list[str]) -> pd.DataFrame:
    if df is None:
        return pd.DataFrame({})
    
    df = pd.get_dummies(
            df, columns=["Sex", "Embarked"], dtype=int
        ).drop(noneed_col, axis=1)
    
    if target_col in df:
        df = df.drop(target_col, axis=1)

    df["Cabin"]         = df["Cabin"].fillna("UKN")
    df["Cabin_count"]   = df["Cabin"].apply(lambda x: 0 if x == "UKN" else len(str(x).split()))
    df["Cabin_num"]     = df["Cabin"].apply(lambda x: ord(str(x)[0]))
    df["Sex"]           = df["Sex_female"]
    df["Age"]           = df["Age"].fillna(df["Age"].mean())
    df["Fare"]           = df["Fare"].fillna(df["Fare"].mean())

    df = df.drop("Cabin", axis=1)
    df = df.drop("Sex_female", axis=1)
    df = df.drop("Sex_male", axis=1)
    
    return df

In [ ]:
params:pd.DataFrame = preprocess(TRAINING_DATA, target_col, params_col, noneed_col)
target:pd.Series | None
if TRAINING_DATA is not None:
    target = TRAINING_DATA[target_col]

display(params)

## Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = params
y = target

X1, X2, y1, y2 = train_test_split(X, y, test_size=0.2, random_state=100)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X1, y1)
model.score(X2, y2)

## Testing

In [ ]:
processed:pd.DataFrame
if TESTINGS_DATA is not None:
    processed = preprocess(TESTINGS_DATA, target_col, params_col, noneed_col)
display(processed)
display(processed.info())

In [ ]:
assert params.columns.to_list() == processed.columns.to_list()

In [ ]:
result:pd.DataFrame
if TESTINGS_DATA is not None:
    result = TESTINGS_DATA
result["Survived"] = model.predict(processed)
result = result[["PassengerId", "Survived"]]

In [ ]:
result.to_csv(final_filepath, index=False)

## Result

0.76076 score